In [33]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tbparse
from IPython.display import display

In [ ]:
# parameters for papermill arguments

root_dir = '.'
results_dir = f'{root_dir}/result_csvs_and_pdfs'
summary_csv_path = f'{results_dir}/summary.csv'

In [ ]:
summary_df = pd.read_csv(summary_csv_path)
display(summary_df)

In [ ]:
summary_df['label'] = summary_df[['model','train_data','test_data', 'param_id']].apply(lambda x:'\n'.join(x), axis=1)
summary_df['percent_epochs'] = summary_df['train_epochs']/summary_df['max_epochs']
summary_df.columns

num_runs = len(summary_df)
num_runs

In [36]:
# get dataframe from lightning log

def get_df_from_log(log_path):
    reader = tbparse.SummaryReader(log_path)
    return reader.scalars

In [37]:
# get dfs from logs

hyperparam_dfs = {}
train_dfs = {}
test_dfs = {}
custom_dfs = {}

for index,row in summary_df.iterrows():
    label_str = row.label
    label = (row.model,row.train_data,row.test_data,row.param_id)
    train_df = get_df_from_log(row.train_llog_path)
    train_dfs[label] = train_df
    test_df = get_df_from_log(row.test_llog_path)
    test_dfs[label] = test_df
    custom_df = get_df_from_log(row.train_clog_path)
    custom_dfs[label] = custom_df
    hyperparam_dfs[label] = {}
    for i in range(row.n_hyperparam_trials):
        hyperparam_dfs[label][i] = get_df_from_log(f'{row.hyper_llog_path}/version_{i}')


In [ ]:
models = ['model', 'train_data', 'test_data', 'param_id']
hyperparams = ['learning_rate', 'batch_size', 'accum_grad_batches', 'max_epochs', 'feature_length', 'dim_mlp_layers']

melt_df = pd.melt(summary_df, id_vars='label', var_name='Metric', value_name='Value')
melt_df = melt_df[melt_df.Metric.isin(hyperparams)]
display(melt_df)

In [ ]:
# barplot hyperparameters combined

df = melt_df.rename(columns={'label': 'Model'})

num_params = len(hyperparams)
plt.figure(figsize=(3 * num_params,8))
sns.barplot(x='Metric', y='Value', hue='Model', data=df, palette='deep')
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.4), ncol=num_runs/2)
plt.xticks(rotation=45)
plt.title('Hyper Parameters by Model')
plt.tight_layout()
plt.savefig(f'{results_dir}/hyperparameter_summary.barplot.pdf')
plt.show()
plt.clf()

In [ ]:
# individual bar plots

x_measures = {
  "Training Runtime (in secs)": "train_walltime",
  "Test AUROC": "test_auroc",
  "Test Loss": "test_loss",
  "Learning Rate": "learning_rate",
  "Batch Size": "batch_size",
  "Gradient Accumulation": "accum_grad_batches",
  "Max Epochs": "max_epochs",
  "Train Epochs": "train_epochs",
  "Train Steps": "train_steps",
  "Percent of Max Epochs Used": "percent_epochs",
}

for x_title,x_col in x_measures.items():
    sns.set(rc={'figure.figsize': (3 * num_runs, 6)})
    sns.barplot(x="label", y=f"{x_col}", data=summary_df, hue="label", palette='deep')
    plt.xlabel("(Model, Training Data, Test Data, Param Set)")
    plt.ylabel(f"{x_title}")
    plt.title(f"{x_title} by Model")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(f'{results_dir}/{x_col}.barplot.pdf')
    plt.show()
    plt.clf()

In [ ]:
# line plots over epochs (using custom logs)

x_measures = {
  "Runtime over Batches": "walltime_per_batch",
  "Loss over Batches": "loss_per_batch",
  "Runtime over Epochs": "walltime_per_epoch",
  "Loss over Epochs": "avgloss_per_epoch",
}

for x_title,x_col in x_measures.items():
    for key,df in custom_dfs.items():
        df_tag = df[df.tag == x_col]
        plt.plot(df_tag.step, df_tag.value, label=key)

    plt.xlabel("Step")
    plt.ylabel(f"{x_title}")
    plt.title(f"{x_title} by Model")
    plt.xticks(rotation=45)

    plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2))
    plt.tight_layout()
    plt.savefig(f'{results_dir}/{x_col}.lineplot.pdf')
    plt.show()
    plt.clf()

In [ ]:
# line plot loss over walltime (using custom logs)

for key,df in custom_dfs.items():
    df_time = df[df.tag == 'walltime_per_batch']
    df_loss = df[df.tag == 'loss_per_batch']
    plt.plot(df_time.value, df_loss.value, label=key)

x_title = "Training Loss over Time"
plt.xlabel("Walltime (in Seconds)")
plt.ylabel(f"{x_title}")
plt.title(f"{x_title} by Model")
plt.xticks(rotation=45)

plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2))
plt.tight_layout()
plt.savefig(f'{results_dir}/loss_per_walltime.lineplot.pdf')
plt.show()
plt.clf()

In [ ]:
# line plots over epochs (using lightning logs)

x_measures = {
  "Training Loss over Epochs": "train_loss_epoch",
  "Training Loss over Steps": "train_loss_step",
  "Positive Prediction Average over Epochs": "pos_prediction_avg",
  "Negative Prediction Average over Epochs": "neg_prediction_avg",
}

for x_title,x_col in x_measures.items():
    for key,df in train_dfs.items():
        df_tag = df[df.tag == x_col]
        plt.plot(df_tag.step, df_tag.value, label=key)

    plt.xlabel("Epochs")
    plt.ylabel(f"{x_title}")
    plt.title(f"{x_title} by Model")
    plt.xticks(rotation=45)

    plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2))
    plt.tight_layout()
    plt.savefig(f'{results_dir}/{x_col}.lineplot.pdf')
    plt.show()
    plt.clf()

In [ ]:
# line plot hyperparam training

x_measures = {
  "Training Loss over Epochs": "train_loss_epoch",
  "Training Loss over Steps": "train_loss_step",
  "Positive Prediction Average over Epochs": "pos_prediction_avg",
  "Negative Prediction Average over Epochs": "neg_prediction_avg",
}

for x_title,x_col in x_measures.items():
    for label,hp_dict in hyperparam_dfs.items():
      label_str = ','.join(label)
      for key,df in hp_dict.items():
          df_tag = df[df.tag == x_col]
          plt.plot(df_tag.step, df_tag.value, label=key)

      plt.xlabel("Epochs")
      plt.ylabel(f"{x_title}")
      plt.title(f"{x_title} by Model\n{label_str}")
      plt.xticks(rotation=45)

      plt.legend(loc="upper right")
      plt.tight_layout()
      plt.savefig(f'{results_dir}/hyperparameter_opt.{x_col}.lineplot.pdf')
      plt.show()
      plt.clf()